In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from PIL import Image

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torchvision import transforms

In [ ]:
PROJECT_ROOT = Path.cwd().parent

UTA_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "sequences"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "checkpoints"
    / "best_cnn.pth"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
)

RESULTS_DIR.mkdir(
    exist_ok=True
)

In [ ]:
all_subjects = sorted(
    [p.name for p in UTA_ROOT.iterdir() if p.is_dir()]
)

print(len(all_subjects))
print(all_subjects)

In [ ]:
TEST_SUBJECTS = [
    "01",
    "06",
    "11",
    "16",
    "21",
    "26",
    "31",
    "36",
    "41",
    "46"
]

TRAIN_SUBJECTS = [
    s for s in all_subjects
    if s not in TEST_SUBJECTS
]

print(len(TRAIN_SUBJECTS))

In [ ]:
print("Train:", len(TRAIN_SUBJECTS))
print("Test :", len(TEST_SUBJECTS))
print(TRAIN_SUBJECTS)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.efficientnet_b0(
    weights=None
)

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 2)
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(checkpoint)
model = model.to(device)
model.eval()

print(model)

In [ ]:
feature_extractor = nn.Sequential(
    model.features,
    model.avgpool,
    nn.Flatten(),
    model.classifier[0],   # Dropout
    model.classifier[1],   # Linear 1280→512
    model.classifier[2]    # ReLU
).to(device)

feature_extractor.eval()

In [ ]:
x = torch.randn(
    1,
    3,
    224,
    224
).to(device)

with torch.no_grad():
    emb = feature_extractor(x)

print(emb.shape)

In [ ]:
transform = models.EfficientNet_B0_Weights.DEFAULT.transforms()

In [ ]:
from pathlib import Path
import random

random.seed(42)

samples = {
    "0": [],
    "5": [],
    "10": []
}

for subject in TRAIN_SUBJECTS:
    subject_dir = UTA_ROOT / subject

    for cls in ["0", "5", "10"]:
        frame_paths = list(
            (subject_dir / cls).glob("*.jpg")
        )

        samples[cls].extend(frame_paths)

sampled_frames = {}

for cls in ["0", "5", "10"]:
    sampled_frames[cls] = random.sample(
        samples[cls],
        166
    )

for cls in sampled_frames:
    print(cls, len(sampled_frames[cls]))

In [ ]:
all_embeddings = []
all_labels = []
all_subjects = []

label_map = {
    "0": "Alert",
    "5": "Low Vigilant",
    "10": "Drowsy"
}

for cls in ["0", "5", "10"]:
    print(f"Processing class {cls}...")

    for img_path in sampled_frames[cls]:

        img = Image.open(img_path).convert("RGB")
        x = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            emb = feature_extractor(x)

        emb = emb.squeeze().cpu().numpy()

        all_embeddings.append(emb)
        all_labels.append(label_map[cls])

        # subject id is the parent of 0/5/10
        subject_id = img_path.parent.parent.name
        all_subjects.append(subject_id)

In [ ]:
X = np.array(all_embeddings)
y = np.array(all_labels)
subjects = np.array(all_subjects)

print(X.shape)
print(y.shape)
print(subjects.shape)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=30,
    random_state=42,
    init="pca"
)

X_tsne = tsne.fit_transform(X_scaled)

print(X_tsne.shape)

In [ ]:
df_tsne = pd.DataFrame({
    "x": X_tsne[:, 0],
    "y": X_tsne[:, 1],
    "class": y,
    "subject": subjects
})

In [ ]:
plt.figure(figsize=(10,8))

sns.scatterplot(
    data=df_tsne,
    x="x",
    y="y",
    hue="class",
    palette={
        "Alert": "green",
        "Low Vigilant": "orange",
        "Drowsy": "red"
    },
    alpha=0.8
)

plt.title("t-SNE of CNN Embeddings by Class")
plt.show()

In [ ]:
plt.figure(figsize=(12,10))

sns.scatterplot(
    data=df_tsne,
    x="x",
    y="y",
    hue="subject",
    palette="tab20",
    legend=False,
    alpha=0.8
)

plt.title("t-SNE of CNN Embeddings by Subject")
plt.show()

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(18,8)
)

sns.scatterplot(
    data=df_tsne,
    x="x",
    y="y",
    hue="class",
    palette={
        "Alert": "green",
        "Low Vigilant": "orange",
        "Drowsy": "red"
    },
    ax=axes[0]
)

axes[0].set_title("By Class")

sns.scatterplot(
    data=df_tsne,
    x="x",
    y="y",
    hue="subject",
    palette="tab20",
    legend=False,
    ax=axes[1]
)

axes[1].set_title("By Subject")

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "tsne_plot.png",
    dpi=300
)

plt.show()